# Содержание

* [Бизнес–контекст (как его формулировали разработчики)](#бизнес-контекст-(как-его-формулировали-разработчики))
* [Обработка данных](#обработка-данных)
* [Проверка соотношения групп (SRM)](#проверка-соотношение-групп-srm)

# Бизнес–контекст (как его формулировали разработчики)

Cookie Cats — популярная мобильная puzzle-игра от Tactile Entertainment (жанр «connect three»).

По мере прохождения уровней игрок сталкивается с __гейтами__ — точками, где его останавливают и заставляют либо подождать, либо сделать внутриигровую покупку, чтобы продолжить.

Изначально первый гейт стоял на __30-м уровне__.

Команда провела A/B-тест, в котором первый гейт перенесли на __40-й уровень__, и хотела понять, как это повлияет на удержание игроков.
* Control: gate_30
* Treatment: gate_40

Данные: 90 189 игроков, которые установили игру во время эксперимента.

Каждый игрок случайно попадал в одну из двух групп.

Метрики, которые собирали:
* retention_1 — вернулся ли игрок на следующий день после установки
* retention_7 — вернулся ли на 7-й день
* sum_gamerounds — сколько раундов сыграл за первые 14 дней

In [16]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import chisquare, binomtest

In [17]:
df = pd.read_csv('tables/cookie_cats.csv')
df.sample(5)

,userid,version,sum_gamerounds,retention_1,retention_7
17421,1939266,gate_30,13,False,False
56976,6313965,gate_30,278,True,True
41495,4591074,gate_40,1,False,False
31804,3537228,gate_30,20,False,False
67084,7430805,gate_30,177,True,False


# Обработка данных

In [18]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB


Пропусков нет.

In [19]:
df['userid'].duplicated().sum()

np.int64(0)

Дубликатов по userid нет.

In [20]:
df[df['sum_gamerounds'] < 0]['sum_gamerounds'].count()

np.int64(0)

Нет отрицательных чисел в `sum_gamerounds`.

In [21]:
df['version'].unique()

<StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

В колонке `version` только нужные нам значения.

In [22]:
df[df['version'] == 'gate_30']['sum_gamerounds'].describe()

count    44700.000000
mean        52.456264
std        256.716423
min          0.000000
25%          5.000000
50%         17.000000
75%         50.000000
max      49854.000000
Name: sum_gamerounds, dtype: float64

In [23]:
df[df['version'] == 'gate_40']['sum_gamerounds'].describe()

count    45489.000000
mean        51.298776
std        103.294416
min          0.000000
25%          5.000000
50%         16.000000
75%         52.000000
max       2640.000000
Name: sum_gamerounds, dtype: float64

Работа с `sum_gamerounds`

Распределение количества раундов сильно скошено вправо (есть экстремальный выброс 49 854).

Поэтому:
* среднее значение приводим для справки,
* основные выводы и статистические тесты делаем по медиане и с помощью непараметрического критерия (Mann-Whitney U).

Игроков с нулевым количеством раундов не удаляем — это важная часть поведения пользователей.

# Проверка соотношение групп (SRM)

In [46]:
print('Абсолютные значения:\n', df['version'].value_counts().sort_index(), sep='', end='\n')
print('Доли:\n', df['version'].value_counts(normalize=True).sort_index() * 100, sep='', end='\n')
print(
    'Разница пользователей: ',
    abs(df['version'].value_counts().loc['gate_40'] - df['version'].value_counts().loc['gate_30'])
)

Абсолютные значения:
version
gate_30    44700
gate_40    45489
Name: count, dtype: int64
Доли:
version
gate_30    49.562585
gate_40    50.437415
Name: proportion, dtype: float64
Разница пользователей:  789


Группы распределены не равномерно, посмотрим есть ли srm.

In [29]:
# chi-square
observed = df['version'].value_counts().sort_index()
expected = [len(df) / 2, len(df) / 2]

chi2, p_chi = chisquare(f_obs=observed, f_exp=expected)

print("Chi-square SRM test")
print(f"Observed: {observed.to_dict()}")
print(f"Expected: {expected}")
print(f"chi2 = {chi2:.3f}, p-value = {p_chi:.4f}")

# binomial test
n_gate40 = (df['version'] == 'gate_40').sum()
n_total = len(df)
binom_result = binomtest(n_gate40, n_total, p=0.5)
print(f"Binomial test p-value = {binom_result.pvalue:.4f}")

Chi-square SRM test
Observed: {'gate_30': 44700, 'gate_40': 45489}
Expected: [45094.5, 45094.5]
chi2 = 6.902, p-value = 0.0086
Binomial test p-value = 0.0087


Для проверки Sample Ratio Mismatch (SRM) я использую базовый уровень значимости **α = 0.05**, а также проверяю более строгий порог **α = 0.01**. В индустриальной практике для SRM часто применяют сверхстрогий порог (**α = 0.001**), так как это диагностический тест валидности сплитования, позволяющий отсеять случайные колебания на больших выборках и не браковать корректные эксперименты.

В нашем случае получено **p-value = 0.0086** (соотношение групп **49.56% / 50.44%**, разница — 789 пользователей):
* при **α = 0.05** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.01** — дисбаланс статистически значим (SRM обнаружен);
* при **α = 0.001** — дисбаланс не является статистически значимым (тест пройден по индустриальному стандарту).

**Вывод и дальнейшие шаги:** Фиксирую наличие лёгкого дисбаланса. Поскольку по строгому индустриальному порогу (α = 0.001) тест проходит проверку, я продолжаю анализ. Однако все финальные продуктовые выводы будут сопровождаться дисклеймером о возможном смещении выборки.


## Анализ удержания

In [62]:
grouped = (
    df.groupby('version')
    .agg(
        users=('userid', 'count'),
        sum_gamerounds=('sum_gamerounds', 'sum'),
        retention_1=('retention_1', 'mean'),
        retention_7=('retention_7', 'mean'),
        gamerounds_mean=('sum_gamerounds', 'mean'),
        gamerounds_median=('sum_gamerounds', 'median')
    )
)
grouped

,users,sum_gamerounds,retention_1,retention_7,gamerounds_mean,gamerounds_median
version,,,,,,
gate_30,44700,2344795,0.448188,0.190201,52.456264,17.0
gate_40,45489,2333530,0.442283,0.182000,51.298776,16.0


In [63]:
grouped.transpose()

version,gate_30,gate_40
users,4.470000e+04,4.548900e+04
sum_gamerounds,2.344795e+06,2.333530e+06
retention_1,4.481879e-01,4.422827e-01
retention_7,1.902013e-01,1.820000e-01
gamerounds_mean,5.245626e+01,5.129878e+01
gamerounds_median,1.700000e+01,1.600000e+01
